# 07 · 使用内置工具（Built-in Tools）

目标：演示 GitHub Copilot SDK **内置（built-in）工具**的发现、调用与精细化管控。

**心智模型**：除了用户用 `@define_tool` 定义的工具外，CLI 自带一组开箱即用的内建工具（文件读写、shell、grep、glob、web fetch …），LLM 可以直接调用。这些工具通过 `tools.list` RPC 发现，通过 `available_tools` / `excluded_tools` 控制可见集合。

本 Notebook 将覆盖：
1. 列出 CLI 内建工具清单（`tools.list` RPC）
2. 让模型自动使用 `view` 读取文件
3. 让模型用 `glob` + `rg` 搜索代码
4. 让模型用 `bash` 执行 shell 命令（带权限审计）
5. 用 `available_tools` 白名单约束模型
6. 用 `excluded_tools` 黑名单屏蔽危险工具

## 0. 15 个内建工具速查图

CLI 内建工具的实际清单（来自 `tools.list` RPC，**与网上很多老示例不一致**）：

```mermaid
flowchart TB
    LLM(["🧠 LLM"]):::llm

    subgraph Shell["🐚 Shell (5)"]
        T1["bash<br/>一次性 shell"]:::danger
        T2["write_bash<br/>(后台 / 长跑)"]:::shell
        T3["read_bash"]:::shell
        T4["stop_bash"]:::shell
        T5["list_bash"]:::shell
    end
    subgraph Files["📁 文件 (2)"]
        T6["view<br/>(唯一的读)"]:::read
        T7["apply_patch<br/>(唯一的写/改/删<br/>freeform patch 协议)"]:::write
    end
    subgraph Search["🔎 搜索 (2)"]
        T8["rg<br/>(ripgrep)"]:::search
        T9["glob"]:::search
    end
    subgraph Net["🌐 网络 (1)"]
        T10["web_fetch"]:::net
    end
    subgraph Orch["🧭 编排 (4)"]
        T11["report_intent"]:::orch
        T12["skill"]:::orch
        T13["task<br/>(子 agent)"]:::special
        T14["ask_user"]:::orch
    end
    subgraph Meta["📚 元数据 (1)"]
        T15["fetch_copilot_cli_documentation"]:::meta
    end

    LLM --> Shell
    LLM --> Files
    LLM --> Search
    LLM --> Net
    LLM --> Orch
    LLM --> Meta

    classDef llm     fill:#1e3a8a,stroke:#1e3a8a,color:#fff,font-weight:bold
    classDef read    fill:#dbeafe,stroke:#2563eb,color:#1e3a8a
    classDef write   fill:#fed7aa,stroke:#ea580c,color:#7c2d12
    classDef shell   fill:#fee2e2,stroke:#dc2626,color:#7f1d1d
    classDef danger  fill:#dc2626,stroke:#7f1d1d,color:#fff,font-weight:bold
    classDef search  fill:#dcfce7,stroke:#16a34a,color:#14532d
    classDef net     fill:#cffafe,stroke:#0891b2,color:#164e63
    classDef orch    fill:#ede9fe,stroke:#7c3aed,color:#4c1d95
    classDef special fill:#7c3aed,stroke:#4c1d95,color:#fff,font-weight:bold
    classDef meta    fill:#f5f5f4,stroke:#78716c,color:#292524
```

> ⚠️ **重要**：没有 `read_file` / `write_file` / `edit_file`。
> - 要**读**文件 → `view`
> - 要**写/改/删**文件 → `apply_patch`（freeform 协议，见 [README.md](README.md) §5.5）
> - 要执行 shell → `bash`（一次性）或 `write_bash`+`read_bash`+`stop_bash`（后台 / 长跑）

详细 JSON Schema 和调用约定见 [docs/builtin-tools.md](docs/builtin-tools.md)。


## 0. 公共初始化（Provider、Client）

In [ ]:
from pathlib import Path
import os, asyncio, json
from dotenv import load_dotenv
from copilot import CopilotClient
from copilot.session import PermissionHandler
from copilot.generated.session_events import (
    AssistantMessageData, SessionIdleData,
    ToolExecutionStartData, ToolExecutionCompleteData,
    PermissionRequestedData,
)
from copilot.generated.rpc import ToolsListRequest

# 加载 .env 获取 GPT 5.4 端点
load_dotenv()

azure_provider = {
    'type': 'azure',
    'base_url': os.environ['AZURE_OPENAI_ENDPOINT_GPT_5_4'],
    'api_key': os.environ['AZURE_OPENAI_API_KEY_GPT_5_4'],
    'azure': {'api_version': os.getenv('AZURE_OPENAI_API_VERSION_GPT_5_4', '2025-03-01-preview')},
}

MODEL = os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4')
# 使用 notebook 所在目录作为 working_directory（其下有 README.md / 各 .ipynb）
WORK_DIR = str(Path.cwd())
print('Working directory:', WORK_DIR)

## 1. 列出 CLI 内建工具清单

通过底层 `tools.list` RPC 直接拉取当前 CLI 暴露的所有内置工具元数据（名称 + 描述 + JSON Schema）。

In [ ]:
async def list_builtin_tools():
    async with CopilotClient() as client:
        # tools.list 是 server 级 RPC，不需要 session
        tool_list = await client._rpc.tools.list(ToolsListRequest(model=MODEL))
        print(f'共发现 {len(tool_list.tools)} 个内建工具：\n')
        for t in tool_list.tools:
            desc = (t.description or '').strip().splitlines()[0][:80]
            print(f'  • {t.name:<24}  {desc}')
        return tool_list.tools

tools = await list_builtin_tools()

## 2. 让模型自动使用 `view` 读取文件

`view` 是内建的文件查看工具（CLI 端实现），LLM 会自主选用它来读文件。

In [ ]:
async def run_view_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
        ) as session:
            done = asyncio.Event()
            final = {'text': None}

            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        print(f'[🔧 内建工具调用] {d.tool_name}  args={getattr(d, "arguments", None)}')
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send(
                'Use the view tool to read the README.md in the current working directory '
                'and summarize its first section in one sentence.'
            )
            await asyncio.wait_for(done.wait(), timeout=60.0)
            print('\n[💬 最终回复]:', final['text'])

await run_view_demo()

## 3. 让模型用 `glob` + `rg` 搜索代码

内建的 `glob`（按文件名匹配）+ `rg`（ripgrep 全文搜索）组合即可让 agent 在代码库中检索。

In [ ]:
async def run_search_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
        ) as session:
            done = asyncio.Event()
            final = {'text': None}

            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        print(f'[🔧 {d.tool_name}] args={getattr(d, "arguments", None)}')
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send(
                'Use glob to find all .ipynb files under the current working directory, '
                'then use rg to search for the keyword "send_and_wait" inside them. '
                'Report the file paths and line numbers in a bullet list.'
            )
            await asyncio.wait_for(done.wait(), timeout=120.0)
            print('\n[💬 最终回复]:\n', final['text'])

await run_search_demo()

## 4. 让模型用 `bash` 执行 shell 命令（带权限审计）

`bash` 是高权限工具，默认会触发 `on_permission_request` 让你审计每条命令。下面用一个**审计型** permission handler：打印命令、自动放行只读命令，拒绝写入类命令。

In [ ]:
from copilot.session import PermissionRequestResult

READ_ONLY_PREFIXES = ('ls', 'pwd', 'cat', 'echo', 'head', 'tail', 'wc', 'date', 'uname', 'whoami')

def audited_permission(request, invocation):
    """打印权限请求详情 → 只放行只读 shell；其余拒绝。

    handler 签名为 (PermissionRequest, {'session_id': ...})，详见 SDK session.py。
    """
    tool_name = request.tool_name or '?'
    cmd = request.full_command_text or ''
    first_token = cmd.strip().split(' ', 1)[0] if cmd else ''
    is_read_only = first_token in READ_ONLY_PREFIXES
    decision = 'approve-once' if is_read_only else 'reject'
    print(f'[🔐 审计] tool={tool_name}  cmd={cmd!r}  =>  {decision}')
    return PermissionRequestResult(kind=decision)

async def run_bash_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=audited_permission,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
        ) as session:
            done = asyncio.Event()
            final = {'text': None}

            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        print(f'[🔧 {d.tool_name}] args={getattr(d, "arguments", None)}')
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            # 提示模型「分两条单独执行」而不是用 && 合并，方便审计逐条放行
            await session.send(
                'Use bash to run "pwd" (a single command) and then separately run "ls" '
                '(a single command). Report the working directory and listing.'
            )
            await asyncio.wait_for(done.wait(), timeout=120.0)
            print('\n[💬 最终回复]:\n', final['text'])

await run_bash_demo()

## 4.5 创建 / 修改文件：`apply_patch`（GHCP SDK 唯一的写文件工具）+ 缓存型 Bypass

SDK **没有** `write_file` / `edit_file`，所有文件改动都走内建的 `apply_patch` 工具——一种受 Anthropic 启发的 **freeform 文本补丁协议**，能在一次调用里完成「新建 / 修改 / 删除」三类操作。补丁格式形如：

```text
*** Begin Patch
*** Add File: path/to/new.txt
+第一行
+第二行
*** End Patch
```

### 🔑 顺带演示：方案 2「问一次记住一次」缓存型 Bypass

> SDK `PermissionRequestResultKind` 当前只暴露 `approve-once` / `reject` / `user-not-available` / `no-result`，**没有** `approve-for-session` 这类直接的"全程放行"枚举。
>
> 实现 session 级 bypass 的最干净办法：**自己维护一个 `set()` 缓存**，handler 命中缓存就自动 `approve-once`，未命中才弹 `input()` 问真人。等效 VS Code Copilot 那个 *"Always allow for this folder"* 按钮。

**缓存 key 粒度**（决定一次放行的覆盖范围）：

| 粒度 | key | 行为 | 推荐度 |
|---|---|---|---|
| 按精确路径 | `(kind, full_path)` | 每个文件单独问 | 🔒 最严 |
| **按目录** ⭐ | `(kind, dirname)` | 同目录下任意文件共享一次放行 | ✅ 本 cell 采用 |
| 按扩展名 | `(kind, ext)` | 所有 `.py` 共享一次放行 | ⚠️ |
| 按 kind 单独 | `(kind,)` | 任何 write 都放行 = 等于全开 | ❌ |

3 选项交互：
- `y` → 仅本次放行
- `s` → **session 级**放行：把整个 scope（这里是目录）加入缓存
- 其它 → 拒绝

In [ ]:
# 💡 演示：apply_patch + 方案 2（缓存型 session-scope bypass，按【目录】粒度）
import tempfile
from copilot.generated.session_events import PermissionRequestKind

# 用临时目录避免污染仓库
TMP_DIR = tempfile.mkdtemp(prefix='ghcp_apply_patch_')
TARGET_FILE_1 = os.path.join(TMP_DIR, 'hello.py')
TARGET_FILE_2 = os.path.join(TMP_DIR, 'world.py')
print(f'目标文件路径:\n  - {TARGET_FILE_1}\n  - {TARGET_FILE_2}')


# 🔑 方案 2：缓存 key 用 (kind, 目录)。同目录下任意文件写入只需问一次
approved_keys: set[tuple[str, str]] = set()


def _cache_key(request) -> tuple[str, str]:
    """生成缓存 key：(kind, 目录) — 同目录下任意 write 共享一条放行规则。"""
    kind = request.kind.value if request.kind else '?'
    target = request.path or request.file_name or request.full_command_text or '*'
    # 文件类请求用目录维度；shell 命令直接用命令文本
    scope = os.path.dirname(target) if (request.path or request.file_name) else target
    return (kind, scope or '*')


def cached_session_bypass(request, invocation):
    key = _cache_key(request)

    # ① 命中缓存：静默放行
    if key in approved_keys:
        target = request.path or request.file_name or request.full_command_text
        print(f'[⚡ cache-hit] scope={key}  target={target!r}  → approve-once (silent)')
        return PermissionRequestResult(kind='approve-once')

    # ② 未命中：弹 input() 问真人
    print('\n' + '─' * 60)
    print('⚠️  权限请求（首次见到这个 scope）')
    print(f'   kind  = {key[0]}')
    print(f'   scope = {key[1]}')
    if request.path:
        print(f'   path  = {request.path}')
    if request.full_command_text:
        print(f'   cmd   = {request.full_command_text}')
    print('─' * 60)
    answer = input('   👉 [y]es once / [n]o / [s]ession-scope (记住整个 scope): ').strip().lower()

    if answer == 's':
        approved_keys.add(key)
        print(f'   → 加入 session 缓存：{key} 整段 scope 后续不再问\n')
        return PermissionRequestResult(kind='approve-once')
    decision = 'approve-once' if answer in ('y', 'yes') else 'reject'
    print(f'   → 你的裁决: {decision}\n')
    return PermissionRequestResult(kind=decision)


async def run_apply_patch_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=cached_session_bypass,
            model=MODEL,
            provider=azure_provider,
            working_directory=TMP_DIR,
        ) as session:
            done = asyncio.Event()
            final = {'text': None}

            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        print(f'[🔧 {d.tool_name}] called')
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            # ⭐ 让模型创建两个文件：第一次按 [s] 放行整个目录 → 第二次写就静默放行
            await session.send(
                'Use apply_patch to create TWO files in the current working directory:\n'
                '1. "hello.py" with `def greet(): print("hi")`\n'
                '2. "world.py" with `def hello_world(): print("world")`\n'
                'Create them one at a time (two separate apply_patch calls), not as a single patch.'
            )
            await asyncio.wait_for(done.wait(), timeout=300.0)
            print('\n[💬 最终回复]:\n', final['text'])

await run_apply_patch_demo()

# ─── 验证 ─────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('[📊 session 缓存最终内容]:', approved_keys)
print('   (按目录粒度 → 同一个目录只有一条规则；如果你按 [s]，应该只有 1 条)')
print('=' * 60)
for fn in (TARGET_FILE_1, TARGET_FILE_2):
    print(f'\n📄 {fn}:')
    try:
        with open(fn) as f:
            print(f.read())
    except FileNotFoundError:
        print('  (不存在 —— 被拒绝)')

## 5. 用 `available_tools` 白名单约束模型

**只允许**这次会话使用列出的工具集合，其他全部不可见。适合「只读浏览」「最小权限」场景。

In [ ]:
async def run_whitelist_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
            available_tools=['view', 'glob', 'rg'],  # ✨ 仅这三个内建工具
        ) as session:
            response = await session.send_and_wait(
                'List which tools you have available now. Do NOT call any tool, just answer.',
                timeout=60.0,
            )
            if response and hasattr(response.data, 'content'):
                print('\n[💬 白名单模式]:', response.data.content)

await run_whitelist_demo()

### 5.1 硬验证：让模型尝试调用被屏蔽的 `bash`

光听模型「自报家门」不可靠（它会把环境里的 `git` / `curl` / `gh` 也算成"工具"，
还会幻觉出 `multi_tool_use.parallel` 这类平台元工具）。

**真正验证白名单生效的办法**：让模型**主动尝试**调用一个不在白名单里的工具，观察 `ToolExecutionStartData` 事件——
- 如果只看到 `view`/`glob`/`rg` 而看不到 `bash` 事件 → 白名单确实生效（CLI 在 schema 层就没把 `bash` 给模型）
- 如果模型回复说「I do not have access to the bash tool」→ 进一步佐证

In [ ]:
# 💡 让模型主动尝试调 bash，然后看真实事件流里到底有哪些 tool 被调用
async def run_whitelist_hard_verify():
    actual_tool_calls = []

    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
            available_tools=['view', 'glob', 'rg'],
        ) as session:
            done = asyncio.Event()
            final = {'text': None}

            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        actual_tool_calls.append(d.tool_name)
                        print(f'[🔧 真实调用] {d.tool_name}')
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send(
                'Try to use the bash tool to run "echo hello-from-bash". '
                'If you cannot, just say "BASH_BLOCKED" plus a one-sentence reason. '
                'Then use glob to count .ipynb files and report the number.'
            )
            await asyncio.wait_for(done.wait(), timeout=60.0)

    print('\n' + '=' * 60)
    print(f'[📊 真实被调用过的工具集合]: {sorted(set(actual_tool_calls))}')
    print(f'[💬 模型回复]: {final["text"]}')
    print('=' * 60)
    if 'bash' not in actual_tool_calls:
        print('[✅ 验证通过] bash 从未被调用 → 白名单确实在 schema 层屏蔽了它')
    else:
        print('[❌ 验证失败] bash 居然被调用了！')

await run_whitelist_hard_verify()

## 6. 用 `excluded_tools` 黑名单屏蔽危险工具

更常见的场景：保留全部内建工具，但屏蔽 `bash` / `apply_patch` / `write_file` 等高风险写入工具。

In [ ]:
async def run_blacklist_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
            excluded_tools=['bash', 'apply_patch', 'write_file', 'edit_file'],  # ✨ 屏蔽写入
        ) as session:
            response = await session.send_and_wait(
                'Briefly list the tool categories you can still use after the blacklist.',
                timeout=60.0,
            )
            if response and hasattr(response.data, 'content'):
                print('\n[💬 黑名单模式]:', response.data.content)

await run_blacklist_demo()

## 8. 调用 `skill`（接入仓库本地 skill）

把仓库 `skills/` 目录注入 `create_session(skill_directories=...)`，LLM 就能用 `skill` 工具按名调用任意 skill（每个 skill 是带 `SKILL.md` + 脚本的小封装）。

> 仓库已有的 skill：`text-search` / `pdf-search` / `excel-search` / `netlist-search` …
> 调用约定：模型只传 skill 名（不传参数），具体逻辑由 skill 内部处理。

In [ ]:
# 💡 演示：把本地 skills/ 目录注入 session，让 LLM 通过 skill 工具调用 text-search
SKILL_DIR = str(Path.cwd() / 'skills')

async def run_skill_demo():
    skill_invocations = []

    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
            skill_directories=[SKILL_DIR],   # ✨ 关键：注入本地 skill 目录
        ) as session:
            done = asyncio.Event()
            final = {'text': None}

            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        if d.tool_name == 'skill':
                            args = getattr(d, 'arguments', None) or {}
                            skill_invocations.append(args.get('skill', '?'))
                            print(f'[🧩 skill 调用] -> {args.get("skill", "?")}')
                        else:
                            print(f'[🔧 {d.tool_name}]')
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send(
                'Use the text-search skill to find any .md file in the current '
                'working directory containing the keyword "send_and_wait". '
                'Then summarize the result in one sentence.'
            )
            await asyncio.wait_for(done.wait(), timeout=120.0)

    print(f'\n[📊 实际触发的 skill 列表]: {skill_invocations}')
    print(f'[💬 最终回复]:\n{final["text"]}')

await run_skill_demo()

### 8.1 验证：skill 调用 → 真的去跑了 skill 内的脚本吗？

上一个 cell 只观察了 `skill` 工具被调用。但 `skill` 工具本身**只加载 SKILL.md 说明书**，真正的脚本执行（`python text_search.py ...`）是 LLM 后续**主动**用 `bash` 去跑的。

下面这个 cell 把整条链路打印出来：
1. `skill` 工具被调用 → CLI 把 [`SKILL.md`](skills/text-search/SKILL.md) 内容塞进 LLM context
2. LLM 阅读后**自主决策**：用 `bash` 跑 `python text_search.py search-tree "..." --root .`
3. `bash` 工具拿到 shell 命令 → 实际 `subprocess.Popen` 执行 → 返回 stdout
4. LLM 读 stdout → 给出最终人类可读答案

我们会捕获每个工具的**完整 args** 和**完整 result**，亲眼看到 python 脚本是怎么被点亮的。

In [ ]:
# 💡 验证：skill → bash → python 脚本 完整执行链
# 抓取每个工具的 args 和 result，看 python text_search.py 是怎么被点亮的

async def run_skill_chain_demo():
    chain = []      # (step_no, tool_name, kind, payload)

    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
            skill_directories=[SKILL_DIR],
        ) as session:
            done = asyncio.Event()
            final = {'text': None}
            counter = {'n': 0}

            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        if d.tool_name in ('skill', 'bash'):
                            counter['n'] += 1
                            args = getattr(d, 'arguments', None) or {}
                            chain.append((counter['n'], d.tool_name, 'CALL', args))
                            print(f'\n[Step {counter["n"]}] 🔧 调用 {d.tool_name}')
                            for k, v in args.items():
                                vs = (v[:200] + '…') if isinstance(v, str) and len(v) > 200 else v
                                print(f'   args.{k} = {vs!r}')
                    case ToolExecutionCompleteData() as d:
                        # 把对应工具的 result 也抓出来（content 是工具执行的真实输出/返回值）
                        if d.result and getattr(d.result, 'content', None):
                            text = d.result.content
                            preview = (text[:400] + '… [+省略]') if len(text) > 400 else text
                            print(f'   ↳ result.content ({len(text)} chars):')
                            for line in preview.splitlines()[:8]:
                                print(f'      {line}')
                            chain.append((counter['n'], '↳', 'RESULT', text))
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send(
                'Use the text-search skill to find any file containing the word '
                '"asyncio" in the current working directory. '
                'Report just the file paths (top 5) in a bullet list.'
            )
            await asyncio.wait_for(done.wait(), timeout=180.0)

    # ─── 汇总 ─────────────────────────────────────────────────────────
    print('\n' + '=' * 60)
    print('[📊 完整链路总结]')
    print('=' * 60)
    tool_seq = [name for _, name, kind, _ in chain if kind == 'CALL']
    print(f'   工具调用顺序: {tool_seq}')
    # 看 skill → bash 是否真的连成一条流水线
    has_skill = 'skill' in tool_seq
    has_bash = 'bash' in tool_seq
    if has_skill and has_bash:
        skill_idx = tool_seq.index('skill')
        bash_idx = tool_seq.index('bash')
        if skill_idx < bash_idx:
            print('   ✅ 链路证明：skill (加载说明书) → bash (执行 python 脚本)')
    if has_bash:
        bash_args = next((a for n, t, k, a in chain if t == 'bash' and k == 'CALL'), None)
        if bash_args:
            print(f'   📜 实际执行的 shell 命令: {bash_args.get("command", "?")}')
    print(f'\n[💬 LLM 最终回复]:\n{final["text"]}')

await run_skill_chain_demo()

## 9. 并行工具调用（`multi_tool_use.parallel` / batched tool calls）

LLM 一轮回复可以**同时**返回多个 tool call，SDK 会**并发**执行它们（不阻塞）。在事件流里表现为多个 `ToolExecutionStartData` 事件背靠背触发，但 `ToolExecutionCompleteData` 的到达顺序取决于各自完成时间。

> 这就是 GPT 工具调用 schema 里那个 `multi_tool_use.parallel` 元工具的作用——属于平台行为，无需开发者操心，只要提示模型「**同时**做多件独立的事」即可。

In [ ]:
# 💡 演示：让 LLM 一次性并发执行多个独立的 glob/rg
import time

async def run_parallel_tools_demo():
    timeline = []  # 记录每个事件的时间戳

    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
        ) as session:
            done = asyncio.Event()
            final = {'text': None}
            t0 = time.time()

            def on_event(event):
                ts = time.time() - t0
                match event.data:
                    case ToolExecutionStartData() as d:
                        timeline.append((ts, 'START', d.tool_name, d.tool_call_id[:8]))
                    case ToolExecutionCompleteData() as d:
                        timeline.append((ts, 'DONE ', '       ', d.tool_call_id[:8]))
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send(
                'In a SINGLE turn, run these THREE independent searches IN PARALLEL '
                '(do not chain them sequentially):\n'
                '  1. glob for "**/*.ipynb"\n'
                '  2. glob for "**/*.md"\n'
                '  3. rg pattern "send_and_wait" with output_mode=count\n'
                'Then briefly summarize all three results.'
            )
            await asyncio.wait_for(done.wait(), timeout=120.0)

    print('\n[⏱️  事件时间线 (秒, 事件, 工具, call_id 前 8 位)]:')
    for ts, evt, name, cid in timeline:
        print(f'  t+{ts:6.3f}s  {evt}  {name:8s}  {cid}')

    print(f'\n[💬 最终回复]:\n{final["text"]}')

await run_parallel_tools_demo()

## 10. 异步 `bash` 会话全生命周期（`bash` + `read_bash` + `list_bash` + `stop_bash`）

`bash` 工具有 `sync` / `async` 两种模式：
- **sync**：等命令完成才返回；会话结束后销毁
- **async**：立即返回 `shellId`，命令在后台继续跑

对长跑命令 / 服务进程 / TUI 程序，必须用 async：
- 用 `read_bash(shellId)` 持续拉取累积输出
- 用 `write_bash(shellId, input=...)` 发送按键（含 `{enter}` `{up}` 等）
- 用 `list_bash()` 列出所有活动会话
- 用 `stop_bash(shellId)` 终止整个会话

In [ ]:
# 💡 演示：模拟一个"长跑"命令（用 sleep + 多次 echo），观察 bash/read_bash/list_bash/stop_bash 协同
async def run_async_bash_demo():
    bash_tool_seq = []

    # 此处全部放行（包括 bash 类）
    def approve_bash(request, invocation):
        return PermissionRequestResult(kind='approve-once')

    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=approve_bash,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
        ) as session:
            done = asyncio.Event()
            final = {'text': None}

            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        if d.tool_name in ('bash', 'read_bash', 'write_bash', 'list_bash', 'stop_bash'):
                            bash_tool_seq.append(d.tool_name)
                            args = getattr(d, 'arguments', None) or {}
                            short = {k: (v[:40] + '…' if isinstance(v, str) and len(v) > 40 else v)
                                     for k, v in args.items()}
                            print(f'[🔧 {d.tool_name:10s}] {short}')
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send(
                'Demonstrate the full async bash lifecycle as follows:\n'
                '1. Start (mode="async") a long-running script: '
                '`for i in 1 2 3 4 5; do echo step $i; sleep 1; done`. '
                'Use shellId="demo-long".\n'
                '2. After starting, call list_bash to confirm the session is alive.\n'
                '3. Wait a moment with read_bash (delay=10, shellId="demo-long") to read accumulated output.\n'
                '4. Call stop_bash on shellId="demo-long" to terminate it.\n'
                'Report the steps you executed and what you observed in 3-4 lines.'
            )
            await asyncio.wait_for(done.wait(), timeout=180.0)

    print(f'\n[📊 bash 系列工具调用序列]: {bash_tool_seq}')
    print(f'[💬 最终回复]:\n{final["text"]}')

await run_async_bash_demo()

## 11. `web_fetch`：抓取 URL → markdown

抓取 HTML 网页并转换成精简 markdown（默认）或返回原始 HTML（`raw=true`）。触发 `PermissionRequestKind.URL`。

In [ ]:
# 💡 演示：让 LLM 用 web_fetch 抓 example.com 并提取页面标题
async def run_web_fetch_demo():
    fetched_urls = []

    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
        ) as session:
            done = asyncio.Event()
            final = {'text': None}

            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        if d.tool_name == 'web_fetch':
                            args = getattr(d, 'arguments', None) or {}
                            fetched_urls.append(args.get('url', '?'))
                            print(
                                f'[🌐 web_fetch] url={args.get("url")}  max_length={args.get("max_length")}  raw={args.get("raw")}')
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send(
                'Use web_fetch to retrieve "https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/deploy-hosted-agent" as markdown'
            )
            await asyncio.wait_for(done.wait(), timeout=60.0)

    print(f'\n[📊 实际抓取的 URL]: {fetched_urls}')
    print(f'[💬 最终回复]:\n{final["text"]}')

await run_web_fetch_demo()

## 12. `task`：在独立 context window 中启动子 agent

`task` 让主 agent 调起一个**专门子 agent**（独立 context），结果一次性回传——既能保持主对话上下文整洁，又能利用专门 agent 的特长。

可选 `agent_type`：
| type | 模型 | 适用 |
|---|---|---|
| `explore` | Haiku | 并行扫码 / 跨模块研究 / 多个无关问题 |
| `task` | Haiku | 跑测试/build/lint：成功只回摘要，失败回完整堆栈 |
| `general-purpose` | Sonnet | 复杂多步任务 + 高质量推理 |
| `code-review` | — | 代码评审，只报 bug/安全/逻辑（**只读不改**）|

> ⚠️ 子 agent **顺序**执行（不并行），**无状态**——必须在 `prompt` 提供完整上下文。

In [ ]:
# 💡 演示：用 task(agent_type="explore") 子 agent 在仓库中跨文件研究并回报
async def run_subagent_demo():
    subagent_calls = []

    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
            working_directory=WORK_DIR,
        ) as session:
            done = asyncio.Event()
            final = {'text': None}

            def on_event(event):
                match event.data:
                    case ToolExecutionStartData() as d:
                        if d.tool_name == 'task':
                            args = getattr(d, 'arguments', None) or {}
                            subagent_calls.append({
                                'agent_type': args.get('agent_type'),
                                'name': args.get('name'),
                                'description': args.get('description'),
                            })
                            print(f'[🤖 task] type={args.get("agent_type")}  name={args.get("name")!r}  desc={args.get("description")!r}')
                    case AssistantMessageData() as d:
                        if d.content:
                            final['text'] = d.content
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send(
                'Spawn an "explore" sub-agent (name="notebook-mapper") with a clear, '
                'self-contained prompt asking it to: list every notebook (.ipynb) in '
                'the current working directory and return a one-line topic summary '
                'for each notebook (e.g. "01_sdk_quickstart.ipynb: install & first run"). '
                'When it returns, present its results verbatim as a bullet list.'
            )
            await asyncio.wait_for(done.wait(), timeout=180.0)

    print(f'\n[📊 task 调用统计]: {len(subagent_calls)} 次')
    for c in subagent_calls:
        print(f'   - {c}')
    print(f'\n[💬 主 agent 最终回复]:\n{final["text"]}')

await run_subagent_demo()

## 13. Takeaways

### 工具发现 & 调用
- **发现**：`client._rpc.tools.list(ToolsListRequest(model=...))` 拉取清单 + JSON Schema（共 **15** 个内建工具）
- **使用**：无需注册，`create_session(working_directory=...)` 后 LLM 自主选用 `view` / `glob` / `rg` / `bash` / `apply_patch` 等
- **写文件**：唯一通道是 `apply_patch`（freeform 文本补丁；没有 `write_file` / `edit_file`）
- **真假工具**：模型自报「可用工具」会混入 shell 命令（`git`/`curl`/`gh`）和元工具（`multi_tool_use.parallel`）。**真相**来自 `tools.list` RPC + `ToolExecutionStartData` 事件流

### 高级编排
- **并行调用**：模型一轮回复中返回多个 tool call → SDK **并发**执行（事件流里 START 时间戳一致，DONE 顺序按完成时间）
- **异步 bash**：`mode="async"` + `shellId` → `list_bash` 查活 + `read_bash` 续读 + `write_bash` 发输入 + `stop_bash` 终止；服务/daemon 用 `detach=true` 让进程脱离 session 存活
- **skill 调用**：`create_session(skill_directories=[...])` 注入本地 skill 目录 → 模型用 `skill` 工具按名调用
- **子 agent**：`task(agent_type=...)` 启动 explore / task / general-purpose / code-review 子 agent，独立 context；**顺序** + **无状态**，必须在 prompt 提供完整上下文

### 权限 & 管控
- **审计**：`on_permission_request(request, invocation)` 按 `request.kind`（`WRITE` / `SHELL` / `READ` / `URL` / `MCP` / `MEMORY` / `CUSTOM_TOOL` / `HOOK`）放行；返回 `PermissionRequestResult(kind='approve-once' / 'reject' / ...)`
- **白名单**：`available_tools=[...]` —— 仅暴露列表内的**内建**工具（不影响自定义工具）
- **黑名单**：`excluded_tools=[...]` —— 屏蔽指定**内建**工具
- **硬验证**：让模型尝试调被屏蔽的工具，看 `ToolExecutionStartData` 事件中是否出现 → 真相验证
- **覆写**：`@define_tool(name='view', overrides_built_in_tool=True)` 显式 opt-in，否则会因命名冲突抛错
- **跳权限**：`@define_tool(skip_permission=True)` 用于只读 / 安全工具

### 🚦 Bypass 速查表
SDK 0.3.0 的 `PermissionRequestResultKind` 仅暴露 4 个值（`approve-once` / `reject` / `user-not-available` / `no-result`），**没** `approve-for-session` 等高级枚举。3 种实战方案：

| 方案 | 适用 | 关键代码 |
|---|---|---|
| **1. 全 session bypass**（暴力）| 脚本 / CI / 内网工具 | `on_permission_request=PermissionHandler.approve_all` |
| **2. 缓存型 session-scope bypass**（推荐）| 桌面 agent / 用户产品 | 自维护 `set()` 缓存（参见 §4.5）|
| **3. 硬塞 RPC 枚举字符串**（hack）| 不推荐 | `PermissionRequestResult(kind='approve-for-session')  # type: ignore` |

### 完整内建工具地图（参见 [`docs/builtin-tools.md`](docs/builtin-tools.md)）

| 类别 | 工具 | 本 notebook 章节 |
|---|---|---|
| 文件读 | `view` | §2 |
| 文件写 | `apply_patch` ⭐ | §4.5 |
| 搜索 | `rg`, `glob` | §3 / §9 |
| Shell | `bash`（sync）| §4 |
| Shell async | `bash` + `read_bash` + `list_bash` + `stop_bash` + `write_bash` | §10 |
| 网络 | `web_fetch` | §11 |
| 编排 | `skill` | §8 |
| 编排 | `task`（子 agent）| §12 |
| 元 | `report_intent`, `fetch_copilot_cli_documentation` | 在前述每节中被模型隐式调用 |
| 交互 | `ask_user` | 见 `04_permissions_and_hooks.ipynb` |